# Análise de Eficiência Energética em Sistema de Refrigeração Industrial

Este notebook apresenta a análise exploratória do desempenho energético de um sistema de refrigeração industrial a partir de dados já tratados na camada `silver`.

O foco está na interpretação do comportamento do COP ao longo do tempo, na identificação de eventos operacionais anômalos e na relação entre eficiência energética e variáveis de processo como vazão, potência e diferença de temperatura.

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

LAKEHOUSE_PATH = "/Volumes/analytics/digital_twin/data"
# LOCALMENTE
# LAKEHOUSE_PATH = "../data"

SILVER_PATH = f"{LAKEHOUSE_PATH}/silver"
input_file = f"{SILVER_PATH}/dados_refrigeracao_enriched.csv"

df = pd.read_csv(input_file)
df["timestamp"] = pd.to_datetime(df["timestamp"])

# usar apenas dados fisicamente válidos
df_analysis = df.loc[~df["flag_invalido"]].copy()
df_analysis = df_analysis.sort_values("timestamp")
df_analysis.head()

## Relatório de arquivos enválidos:

In [0]:
total = len(df)
invalid = df["flag_invalido"].sum()
pct_invalid = invalid / total * 100

resumo_flags = pd.Series({
    "Total de linhas": total,
    "Linhas inválidas": invalid,
    "% inválidas": pct_invalid,
    "Vazão inválida": df["flag_vazao_invalida"].sum(),
    "Potência inválida": df["flag_potencia_invalida"].sum(),
    "ΔT inválido": df["flag_dT_invalido"].sum(),
    "Temp invertida": df["flag_temp_invertida"].sum(),
})

resumo_flags


## COP

In [0]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(df_analysis["cop"], bins=40, edgecolor="black")
ax.set_title("Distribuição do COP")
ax.set_xlabel("COP (-)")
ax.set_ylabel("Frequência")
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

A distribuição do COP mostra que o sistema opera predominantemente em uma faixa bem definida, indicando um regime operacional relativamente estável.

A presença de valores mais baixos, ainda que pouco frequentes, sugere a ocorrência de eventos pontuais de degradação de eficiência, que merecem investigação ao longo da análise temporal.

In [0]:
cop_stats = df["cop"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
cop_stats

A análise estatística reforça a estabilidade do sistema, com média e mediana próximas, indicando ausência de assimetria significativa na distribuição.

Os percentis mostram que a maior parte da operação se concentra em uma faixa estreita de desempenho, enquanto valores mínimos mais baixos indicam eventos anômalos que fogem do comportamento típico do sistema.

In [0]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(df_analysis["timestamp"], df_analysis["cop"], linewidth=1.0, label="COP")
ax.axhline(df_analysis["cop"].mean(), linestyle="--", linewidth=1, label="COP médio")

ax.set_title("Evolução Temporal do COP")
ax.set_xlabel("Data")
ax.set_ylabel("COP (-)")

ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))

ax.grid(True, linestyle="--", alpha=0.4)
ax.legend()

# rotação dos labels
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.show()

A série temporal do COP evidencia um comportamento globalmente estável, com oscilações ao longo do tempo e quedas pontuais de eficiência.

Observa-se maior variabilidade no início do período analisado, seguida por uma redução progressiva das oscilações, sugerindo a transição do sistema para um regime operacional mais estável.

In [0]:
df_analysis["cop_std_rolling"] = df_analysis["cop"].rolling(window=50).std()

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(df_analysis["timestamp"], df_analysis["cop_std_rolling"], linewidth=1.0)

ax.set_title("Variabilidade do COP (Desvio Padrão Móvel)")
ax.set_xlabel("Data")
ax.set_ylabel("Desvio padrão")

# padronização do eixo de tempo
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))

# grid padrão do projeto
ax.grid(True, linestyle="--", alpha=0.4)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.show()

A análise do desvio padrão móvel confirma a redução da variabilidade do COP ao longo do tempo.

No início do período, o sistema apresenta maior dispersão, enquanto nos dias seguintes a variabilidade se estabiliza em níveis mais baixos (entre aproximadamente 0,1 e 0,2).

Esse comportamento é consistente com a entrada do sistema em regime estacionário, no qual as condições operacionais passam a variar menos e o desempenho se torna mais previsível.

## Vazão ao longo do tempo

In [0]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(df_analysis["timestamp"], df_analysis["vazao_m3_h"], linewidth=1.0)

ax.set_title("Vazão de Água ao Longo do Tempo")
ax.set_xlabel("Data")
ax.set_ylabel("Vazão (m³/h)")

ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))

ax.grid(True, linestyle="--", alpha=0.4)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

A vazão apresenta comportamento predominantemente estável, com oscilações suaves ao longo do tempo e quedas pontuais bem definidas.

Essas reduções de vazão são particularmente relevantes, pois impactam diretamente a capacidade de remoção de calor do sistema, podendo explicar eventos de queda no COP observados anteriormente.

## ΔT ao longo do tempo

In [0]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(df_analysis["timestamp"], df_analysis["dT_c"], linewidth=1.0)

ax.set_title("Diferença de Temperatura ao Longo do Tempo")
ax.set_xlabel("Data")
ax.set_ylabel("ΔT (°C)")

ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))

ax.grid(True, linestyle="--", alpha=0.4)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

A diferença de temperatura (ΔT) reflete a eficiência da troca térmica no sistema.

Variações ao longo do tempo indicam mudanças nas condições de operação, sendo que reduções no ΔT podem estar associadas a menor eficiência na remoção de calor, impactando negativamente o desempenho energético.

## Potência ao longo do tempo

In [0]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(df_analysis["timestamp"], df_analysis["potencia_kw"], linewidth=1.0)

ax.set_title("Potência Elétrica ao Longo do Tempo")
ax.set_xlabel("Data")
ax.set_ylabel("Potência (kW)")

ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))

ax.grid(True, linestyle="--", alpha=0.4)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

A potência elétrica apresenta variações relativamente controladas ao longo do período analisado.

Quando analisada em conjunto com o COP, essa variável permite identificar situações em que o sistema consome mais energia sem ganho proporcional na remoção de calor, caracterizando perda de eficiência operacional.

Até este ponto, foram analisadas as variáveis de forma individual ao longo do tempo.

A partir daqui, a análise passa a focar na relação entre o COP e as variáveis operacionais, com o objetivo de identificar os fatores que mais influenciam a eficiência do sistema.

## Relação entre COP e Variáveis Operacionais

Até este ponto, as variáveis foram analisadas individualmente ao longo do tempo.

Nesta etapa, o foco passa a ser a relação entre o COP e as principais variáveis operacionais do sistema, com o objetivo de identificar quais fatores mais influenciam a eficiência energética do chiller.

In [0]:
plt.figure(figsize=(6,4))
plt.scatter(df_analysis["vazao_m3_h"], df_analysis["cop"], alpha=0.4)

plt.title("Relação entre COP e Vazão")
plt.xlabel("Vazão (m³/h)")
plt.ylabel("COP")

plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

Observa-se uma relação positiva entre vazão e COP, indicando que maiores taxas de fluxo estão associadas a melhor eficiência do sistema.

Além disso, identifica-se a presença de dois regimes operacionais distintos: um regime principal, com maior vazão e maior eficiência, e um regime degradado, caracterizado por redução significativa da vazão e queda acentuada do COP.

Esse comportamento sugere que limitações no fluxo de água impactam diretamente a capacidade de troca térmica, reduzindo a eficiência do sistema.

In [0]:
plt.figure(figsize=(6,4))
plt.scatter(df_analysis["dT_c"], df_analysis["cop"], alpha=0.4)

plt.title("Relação entre COP e ΔT")
plt.xlabel("ΔT (°C)")
plt.ylabel("COP")

plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

A relação entre COP e ΔT mostra que, mesmo nos pontos de menor eficiência, o ΔT permanece dentro de uma faixa operacional relativamente estável.

Esse comportamento indica que a queda do COP não está diretamente associada a uma perda significativa na diferença de temperatura, sugerindo que a degradação da eficiência pode estar relacionada a outros fatores, como redução de vazão ou aumento no consumo energético.

In [0]:
plt.figure(figsize=(6,4))
plt.scatter(df_analysis["potencia_kw"], df_analysis["cop"], alpha=0.4)

plt.title("Relação entre COP e Potência")
plt.xlabel("Potência (kW)")
plt.ylabel("COP")

plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

A relação entre potência e COP indica que o sistema mantém níveis de consumo energético relativamente estáveis mesmo em condições de menor eficiência.

Observa-se a presença de pontos com baixo COP dentro da faixa normal de potência, evidenciando situações em que o sistema consome energia sem conversão eficiente em capacidade de refrigeração.

Esse comportamento caracteriza condições de ineficiência operacional, possivelmente associadas a limitações de vazão ou desbalanceamento do sistema.

## Outras variáveis (Pressão do óleo e Nível do tanque de água)

In [0]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_analysis["timestamp"], df_analysis["pressao_oleo_bar"], linewidth=1.0)

ax.set_title("Pressão do Óleo ao Longo do Tempo")
ax.set_xlabel("Data")
ax.set_ylabel("Pressão (bar)")

ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))

ax.grid(True, linestyle="--", alpha=0.4)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.scatter(df_analysis["pressao_oleo_bar"], df_analysis["cop"], alpha=0.4)

plt.title("Relação entre COP e Pressão do Óleo")
plt.xlabel("Pressão do Óleo (bar)")
plt.ylabel("COP")

plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

A relação entre COP e pressão de óleo não apresenta uma correlação clara.

A maior parte dos pontos se concentra em uma faixa relativamente estreita de pressão, dentro da qual são observados tanto valores altos quanto baixos de COP.

Isso indica que a pressão de óleo não é um fator determinante para as variações de eficiência observadas, sugerindo que o sistema mecânico está operando de forma estável e sem impacto relevante sobre o desempenho energético.

In [0]:
plt.plot(df_analysis["timestamp"], df_analysis["nivel_tanque_pct"], linewidth=1.0)

plt.title("Nível do Tanque ao Longo do Tempo")
plt.xlabel("Data")
plt.ylabel("Nível (%)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.scatter(df_analysis["nivel_tanque_pct"], df_analysis["cop"], alpha=0.4)

plt.title("Relação entre COP e Nível do Tanque")
plt.xlabel("Nível do Tanque (%)")
plt.ylabel("COP")


A relação entre COP e nível do tanque não apresenta uma correlação direta clara.

Embora eventos de menor eficiência estejam associados a determinados níveis, observa-se que diferentes regimes de COP ocorrem dentro de faixas semelhantes de nível.

Isso indica que o nível do tanque, isoladamente, não é suficiente para explicar as variações de eficiência, sugerindo atuação conjunta com outras variáveis do sistema.

In [0]:
plt.scatter(df_analysis["nivel_tanque_pct"], df_analysis["vazao_m3_h"], alpha=0.4)
plt.title("Relação entre Nível do Tanque e Vazão")
plt.xlabel("Nível do Tanque (%)")
plt.ylabel("Vazão (m³/h)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

A relação entre nível do tanque e vazão indica que, mesmo em condições de nível reduzido, o sistema ainda mantém valores de vazão dentro do regime normal.

Esse comportamento sugere que o sistema possui capacidade de compensação hidráulica, mantendo o fluxo mesmo em condições adversas.

No entanto, a persistência de níveis baixos pode representar um risco operacional, podendo levar a instabilidades caso limites críticos sejam atingidos.

## Conclusão Geral

A análise exploratória dos dados operacionais do chiller permitiu identificar os principais fatores associados à eficiência energética do sistema (COP).

Observou-se que a vazão é a variável com maior influência sobre o desempenho, apresentando relação direta com o COP. Reduções na vazão estão associadas a quedas significativas de eficiência, caracterizando eventos operacionais críticos.

A diferença de temperatura (ΔT) permaneceu relativamente estável ao longo do período, indicando que a capacidade de troca térmica não foi o fator determinante para as variações de desempenho observadas.

A potência elétrica apresentou consumo consistente, inclusive em condições de baixa eficiência, evidenciando situações de operação energeticamente ineficiente.

A pressão de óleo não demonstrou correlação relevante com o COP, sugerindo que o sistema mecânico operou de forma estável durante o período analisado.

O nível do tanque apresentou variações ao longo do tempo, incluindo eventos de queda acentuada. No entanto, sua relação com o COP não se mostrou direta, indicando influência indireta ou secundária no desempenho do sistema.

De forma geral, os resultados indicam que a eficiência do chiller depende principalmente do equilíbrio hidráulico do sistema, sendo a manutenção de níveis adequados de vazão o fator crítico para operação eficiente.

### Resumo

A análise mostra que o sistema opera majoritariamente em faixa estável de eficiência, com COP concentrado em torno de ~3,25. As quedas de desempenho estão associadas principalmente à redução de vazão, aumento de instabilidade operacional e alterações no regime térmico, sugerindo que estratégias de monitoramento baseadas em COP, vazão e variabilidade local podem antecipar degradações antes de falhas críticas.